In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
from tqdm import tqdm
import json
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.style.use('dark_background')
print("Using device : ",device)
import albumentations as A
from albumentations.pytorch import ToTensorV2
from accelerate import Accelerator, notebook_launcher
import gc

In [ ]:
# Paths
NOTEBOOK_DIR = Path().resolve()
BASE_DIR = NOTEBOOK_DIR.parents[1]
SEG_DATASET_DIR = BASE_DIR / "data" / "raw" / "kvasir-seg" / "Kvasir-SEG"
INPUT_DIR = SEG_DATASET_DIR / "images"
TARGET_DIR = SEG_DATASET_DIR / "masks"
MODEL_OUTPUT_DIR = BASE_DIR / "models" / "9-pranet-full"
MODEL_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
HEIGHT , WIDTH = 256 , 256
IMAGE_SIZE = (HEIGHT , WIDTH)
import sys
sys.path.append(str(BASE_DIR))
import helper

In [ ]:


HEIGHT, WIDTH = 352, 352
IMAGE_SIZE = (HEIGHT, WIDTH)

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, img_size=(352, 352)):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform
        self.img_size = img_size

        valid_exts = {".jpg", ".jpeg", ".png"}
        self.images = sorted([f for f in self.image_dir.iterdir()
                              if f.suffix.lower() in valid_exts])
        self.masks = sorted([f for f in self.mask_dir.iterdir()
                             if f.suffix.lower() in valid_exts])
        assert len(self.images) == len(self.masks)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img  = np.array(Image.open(self.images[idx]).convert('RGB'))
        mask = np.array(Image.open(self.masks[idx]).convert('L'))

        # Resize
        img  = np.array(Image.fromarray(img).resize(self.img_size, Image.BILINEAR))
        mask = np.array(Image.fromarray(mask).resize(self.img_size, Image.NEAREST))

        # Binarize mask
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']           # Already tensor from ToTensorV2
            mask = aug['mask'].unsqueeze(0)
        
        return img, mask

# 880/120 split — fixed seed for reproducibility
full_dataset = SegmentationDataset(INPUT_DIR, TARGET_DIR, img_size=IMAGE_SIZE)

torch.manual_seed(42)
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [880, 120])

# Assign transforms
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform   = val_transform

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# ─── HarDNet-MSEG Model ───────────────────────────────────────────────────────
# Paper: https://arxiv.org/abs/2101.07172
# Encoder  : HarDNet68 (pretrained on ImageNet)
# Decoder  : Cascaded Partial Decoder — uses only deep features (layers 3,4,5)
#            RFB on each skip, dense aggregation via element-wise multiply
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F

# ── install once if not present ───────────────────────────────────────────────
# !pip install hardnet-pytorch   (or use the authors' repo weight loader below)

# We load HarDNet68 from the official repo via torch.hub
# weights are ImageNet pretrained
def get_hardnet68():
    model = torch.hub.load('PingoLH/Pytorch-HarDNet', 'hardnet68', pretrained=True)
    return model


# ── RFB (Receptive Field Block) ───────────────────────────────────────────────
class RFB(nn.Module):
    """
    Multi-branch dilated conv block from the paper (Figure 4).
    Enlarges receptive field on each skip-connection feature map.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 4

        self.branch0 = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
        )
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=3, dilation=3), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 5, padding=2), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=5, dilation=5), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 7, padding=3), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=7, dilation=7), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
        )
        self.conv_cat = nn.Sequential(
            nn.Conv2d(mid * 4, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
        # residual projection if channels differ
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        out = self.conv_cat(torch.cat([x0, x1, x2, x3], dim=1))
        return out + self.shortcut(x)


# ── Aggregation (element-wise multiply, Figure 5) ─────────────────────────────
class AggregationModule(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x1, x2):
        # upsample x1 to match x2 spatial size, then element-wise multiply
        x1 = F.interpolate(x1, size=x2.shape[2:], mode='bilinear', align_corners=True)
        return self.conv(x1 * x2)


# ── HarDNet-MSEG ─────────────────────────────────────────────────────────────
class HarDNetMSEG(nn.Module):
    """
    Encoder : HarDNet68 — we tap features at blocks 10, 12, 14 (deep only,
              per the Cascaded Partial Decoder design that discards shallow layers)
    Channels : 320, 640, 1024  (HarDNet68 output channels at those blocks)
    Decoder  : RFB on each feature → dense aggregation (multiply) bottom-up
    """
    def __init__(self, pretrained=True):
        super().__init__()

        # ── Encoder ──────────────────────────────────────────────────────────
        backbone = get_hardnet68()

        # HarDNet68 is a Sequential of 'base' blocks.
        # We split it to get intermediate feature maps at 3 deep stages.
        # blocks 0..9  → stage1 (stride 16, ch=320)
        # blocks 0..11 → stage2 (stride 32, ch=640)
        # full          → stage3 (stride 32, ch=1024)
        base = backbone.base  # nn.ModuleList
        self.enc1 = nn.Sequential(*list(base)[:10])   # → 320 ch
        self.enc2 = nn.Sequential(*list(base)[10:12]) # → 640 ch
        self.enc3 = nn.Sequential(*list(base)[12:])   # → 1024 ch

        # channel counts — verified against hardnet68 architecture
        ch1, ch2, ch3 = 320, 640, 1024
        dec_ch = 64

        # ── RFB on each skip ──────────────────────────────────────────────────
        self.rfb1 = RFB(ch1, dec_ch)
        self.rfb2 = RFB(ch2, dec_ch)
        self.rfb3 = RFB(ch3, dec_ch)

        # ── Aggregation: deep → shallow, two steps ────────────────────────────
        self.agg1 = AggregationModule(dec_ch, dec_ch)  # rfb3 × rfb2
        self.agg2 = AggregationModule(dec_ch, dec_ch)  # agg1 × rfb1

        # ── Final prediction head ─────────────────────────────────────────────
        self.head = nn.Conv2d(dec_ch, 1, 1)

        # ── Auxiliary heads (deep supervision — optional but paper uses it) ───
        self.aux3 = nn.Conv2d(dec_ch, 1, 1)
        self.aux2 = nn.Conv2d(dec_ch, 1, 1)

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]

        # Encoder
        f1 = self.enc1(x)   # stride 16
        f2 = self.enc2(f1)  # stride 32
        f3 = self.enc3(f2)  # stride 32

        # RFB
        r1 = self.rfb1(f1)
        r2 = self.rfb2(f2)
        r3 = self.rfb3(f3)

        # Cascaded aggregation (deep → shallow)
        a1 = self.agg1(r3, r2)   # r3 upsampled × r2
        a2 = self.agg2(a1, r1)   # a1 upsampled × r1

        # Upsample to input resolution
        out  = F.interpolate(self.head(a2), size=(H, W), mode='bilinear', align_corners=True)
        aux3 = F.interpolate(self.aux3(r3), size=(H, W), mode='bilinear', align_corners=True)
        aux2 = F.interpolate(self.aux2(a1), size=(H, W), mode='bilinear', align_corners=True)

        if self.training:
            return out, aux2, aux3   # 3 outputs for deep supervision loss
        return out                   # single output at inference


# ── Quick sanity check ────────────────────────────────────────────────────────
if __name__ == "__main__":
    model = HarDNetMSEG(pretrained=True).cuda()
    x = torch.randn(2, 3, 352, 352).cuda()
    model.train()
    out, aux2, aux3 = model(x)
    print(out.shape, aux2.shape, aux3.shape)
    # → torch.Size([2,1,352,352]) ×3

In [ ]:
# ─── Loss Functions for HarDNet-MSEG ─────────────────────────────────────────
# Source: directly borrowed from PraNet (DengPingFan/PraNet) — which is what
# HarDNet-MSEG's training script uses verbatim (the repo is a fork of PraNet).
#
# structure_loss = Weighted BCE + Weighted IoU
#   - weight map (weit) emphasises boundary pixels via avg_pool trick
#   - applied to ALL three outputs: main + aux2 + aux3 (deep supervision)
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn.functional as F


def structure_loss(pred, mask):
    """
    Weighted BCE + Weighted IoU loss (PraNet / HarDNet-MSEG protocol).

    Weight map construction:
      weit = 1 + 5 * |avg_pool(mask) - mask|
      avg_pool with a large kernel (31×31) produces a blurred version of the
      mask. Pixels where the blurred mask differs most from the original are
      boundary pixels — those get up-weighted by up to ×6.
      Interior and background pixels far from any edge stay close to weight 1.

    Args:
        pred : raw logits  (B, 1, H, W)  — NOT sigmoid-ed yet
        mask : binary GT   (B, 1, H, W)  — float32, values in {0, 1}
    Returns:
        scalar loss
    """
    # ── boundary-aware weight map ─────────────────────────────────────────────
    weit = 1 + 5 * torch.abs(
        F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask
    )

    # ── weighted BCE (pixel-level / local) ────────────────────────────────────
    wbce = F.binary_cross_entropy_with_logits(pred, mask, reduction='none')
    wbce = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))

    # ── weighted IoU (region-level / global) ──────────────────────────────────
    pred_sig = torch.sigmoid(pred)
    inter = ((pred_sig * mask) * weit).sum(dim=(2, 3))
    union = ((pred_sig + mask) * weit).sum(dim=(2, 3))
    wiou  = 1 - (inter + 1) / (union - inter + 1)

    return (wbce + wiou).mean()


def calc_loss(pred, mask, aux2, aux3,
              w_main=1.0, w_aux=0.4):
    """
    Deep supervision wrapper — applies structure_loss to all three outputs.

    Weights (same convention as PraNet):
        main output  : w_main = 1.0
        aux outputs  : w_aux  = 0.4  each

    Args:
        pred  : main  output logits (B,1,H,W)
        mask  : GT mask             (B,1,H,W)
        aux2  : auxiliary output 2  (B,1,H,W)
        aux3  : auxiliary output 3  (B,1,H,W)
    Returns:
        total loss (scalar), dict of individual losses for logging
    """
    loss_main = structure_loss(pred, mask)
    loss_aux2 = structure_loss(aux2, mask)
    loss_aux3 = structure_loss(aux3, mask)

    total = w_main * loss_main + w_aux * loss_aux2 + w_aux * loss_aux3

    return total, {
        'loss_main' : loss_main.item(),
        'loss_aux2' : loss_aux2.item(),
        'loss_aux3' : loss_aux3.item(),
        'loss_total': total.item(),
    }

In [ ]:
# Model, loss, optimizer
model = HarDNetMSEG(pretrained=True).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-5)

helper.inspect_model(
    model,
    input_size=(1, 3, HEIGHT, WIDTH),
    criterion=structure_loss,
    optimizer=optimizer,
    model_name="HarDNet-MSEG"
)

In [ ]:
history = {
    'train_loss': [], 'train_main': [], 'train_aux2': [], 'train_iou': [],
    'val_loss':   [], 'val_main':   [], 'val_aux2':   [], 'val_iou':   []
}
best_val_loss = float("inf")

In [ ]:
# ─── Put this in the cell BEFORE the training cell ───────────────────────────

def train_epoch(model, loader, optimizer, accelerator):
    model.train()
    total_loss = total_main = total_aux2 = total_aux3 = total_iou = 0.0

    for imgs, masks in loader:
        with accelerator.accumulate(model):
            optimizer.zero_grad()

            pred, aux2, aux3 = model(imgs)                        # 3 outputs in train mode
            loss, loss_dict  = calc_loss(pred, masks, aux2, aux3) # your existing calc_loss

            accelerator.backward(loss)
            optimizer.step()

        # gather across both GPUs
        total_loss  += accelerator.gather(loss.detach()).mean().item()
        total_main  += loss_dict['loss_main']
        total_aux2  += loss_dict['loss_aux2']
        total_aux3  += loss_dict['loss_aux3']

        # IoU (using sigmoid of main pred)
        with torch.no_grad():
            pred_bin = (torch.sigmoid(pred.detach()) > 0.5).float()
            inter    = (pred_bin * masks).sum(dim=(1,2,3))
            union    = (pred_bin + masks - pred_bin * masks).sum(dim=(1,2,3))
            iou      = accelerator.gather(((inter + 1e-6) / (union + 1e-6))).mean().item()
            total_iou += iou

    n = len(loader)
    return total_loss/n, total_main/n, total_aux2/n, total_iou/n


def validate(model, loader, accelerator):
    model.eval()
    total_loss = total_main = total_aux2 = total_aux3 = total_iou = 0.0

    with torch.no_grad():
        for imgs, masks in loader:
            pred = model(imgs)   # eval mode → single output only

            # use structure_loss directly — no aux outputs in eval
            loss = structure_loss(pred, masks)

            total_loss  += accelerator.gather(loss).mean().item()
            total_main  += accelerator.gather(loss).mean().item()  # main = total in eval
            total_aux2  += 0.0   # no aux in eval, just pad with 0
            total_aux3  += 0.0

            pred_bin = (torch.sigmoid(pred) > 0.5).float()
            inter    = (pred_bin * masks).sum(dim=(1,2,3))
            union    = (pred_bin + masks - pred_bin * masks).sum(dim=(1,2,3))
            iou      = accelerator.gather(((inter + 1e-6) / (union + 1e-6))).mean().item()
            total_iou += iou

    n = len(loader)
    return total_loss/n, total_main/n, total_aux2/n, total_aux3/n, total_iou/n

In [ ]:
# Training with detailed loss tracking
epochs = 25

def training_function():
    accelerator = Accelerator(mixed_precision="fp16")

    # prepare all objects inside
    prepared_model, prepared_optimizer, prepared_train_loader, prepared_val_loader = accelerator.prepare(
        model, optimizer, train_loader, val_loader
    )

    best_val_loss = float("inf")

    gc.collect()
    torch.cuda.empty_cache()

    for epoch in range(epochs):
        accelerator.print(f"Epoch {epoch+1}/{epochs}")

        train_loss, train_dice, train_bce, train_iou = train_epoch(
            prepared_model, prepared_train_loader, prepared_optimizer, accelerator
        )
        val_loss, val_dice, val_bce, val_iou = validate(
            prepared_model, prepared_val_loader, accelerator
        )

        scheduler.step()

        if accelerator.is_main_process:
            # Store history
            history['train_loss'].append(train_loss)
            history['train_main'].append(train_dice)   # train_dice variable = loss_main value
            history['train_aux2'].append(train_bce)    # train_bce variable  = loss_aux2 value
            history['train_iou'].append(train_iou)

            history['val_loss'].append(val_loss)
            history['val_main'].append(val_dice)       # same renaming
            history['val_aux2'].append(val_bce)
            history['val_iou'].append(val_iou)

            # Print
            print(f"TRAIN → Total: {train_loss:.4f} | Main: {train_dice:.4f} | Aux2: {train_bce:.4f} | IoU: {train_iou:.4f}")
            print(f"VAL   → Total: {val_loss:.4f} | Main: {val_dice:.4f} | Aux2: {val_bce:.4f} | IoU: {val_iou:.4f}")
            # Save best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                accelerator.save(
                    accelerator.unwrap_model(prepared_model).state_dict(),
                    MODEL_OUTPUT_DIR / "best_pranet_model.pth"
                )
                print("✓ Best model saved!")

notebook_launcher(training_function, args=(), num_processes=2, mixed_precision="fp16")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 6))

epochs_range = range(1, len(history['train_loss']) + 1)

# Total Loss
axes[0, 0].plot(epochs_range, history['train_loss'], 'o-', color='#00d9ff', linewidth=2, label='Train', markersize=4)
axes[0, 0].plot(epochs_range, history['val_loss'],   's-', color='#ff006e', linewidth=2, label='Val',   markersize=4)
axes[0, 0].set_title('Total Loss (wBCE + wIoU)', fontsize=13, fontweight='bold', pad=12)
axes[0, 0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Loss',  fontsize=11, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3, linestyle='--')

# Aux Losses
axes[0, 1].plot(epochs_range, history['train_main'], 'o-', color='#00d9ff', linewidth=2, label='Main', markersize=4)
axes[0, 1].plot(epochs_range, history['train_aux2'], 's-', color='#ff006e', linewidth=2, label='Aux2', markersize=4)
axes[0, 1].set_title('Train: Per-Head Loss', fontsize=13, fontweight='bold', pad=12)
axes[0, 1].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Loss',  fontsize=11, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3, linestyle='--')

# LR (pulled from history if you log it, else skip)
axes[1, 0].plot(epochs_range, history['train_iou'], 'o-', color='#00d9ff', linewidth=2, label='Train', markersize=4)
axes[1, 0].set_title('Train IoU', fontsize=13, fontweight='bold', pad=12)
axes[1, 0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('IoU Score', fontsize=11, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3, linestyle='--')

# IoU
axes[1, 1].plot(epochs_range, history['train_iou'], 'o-', color='#00d9ff', linewidth=2, label='Train', markersize=4)
axes[1, 1].plot(epochs_range, history['val_iou'],   's-', color='#ff006e', linewidth=2, label='Val',   markersize=4)
axes[1, 1].set_title('IoU Score', fontsize=13, fontweight='bold', pad=12)
axes[1, 1].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('IoU Score', fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(MODEL_OUTPUT_DIR / 'training_history_detailed.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()

In [ ]:
# ─── Inference & Evaluation — HarDNet-MSEG ───────────────────────────────────

# ── 1. Load best model ────────────────────────────────────────────────────────
model.load_state_dict(torch.load(MODEL_OUTPUT_DIR / "best_hardnet_mseg.pth"))
model.eval()

# ── 2. Single sample visualization ───────────────────────────────────────────
image_paths = helper.get_images_from_dirs([INPUT_DIR, TARGET_DIR])

img_pil  = Image.open(image_paths[0]).convert('RGB')
mask_pil = Image.open(image_paths[1]).convert('L')

img_resized  = img_pil.resize((HEIGHT, WIDTH))
mask_resized = mask_pil.resize((HEIGHT, WIDTH))

# ImageNet normalization — must match training transform
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])
img_norm = (np.array(img_resized).astype(np.float32) / 255.0 - mean) / std
img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(img_tensor)                    # eval mode → single output
    pred = torch.sigmoid(pred).cpu().numpy()[0, 0]

img_np   = np.array(img_resized)
mask_np  = np.array(mask_resized) / 255.0

overlay_true = create_overlay(img_np, mask_np)
overlay_pred = create_overlay(img_np, mask_np, pred)

helper.show_grid({
    'True Mask'      : (mask_np * 255).astype(np.uint8),
    'Predicted Mask' : (pred    * 255).astype(np.uint8),
    'RGB + True Mask': overlay_true,
    'RGB + Predicted': overlay_pred
}, grid="row", width=1300)


# ── 3. Evaluation function ────────────────────────────────────────────────────
def evaluate_validation_set(model, val_image_dir, val_mask_dir, device,
                             img_size=(HEIGHT, WIDTH), sample_pixels=10000):
    model.eval()

    val_images = sorted(list(Path(val_image_dir).glob('*.png')) +
                        list(Path(val_image_dir).glob('*.jpg')))
    val_masks  = sorted(list(Path(val_mask_dir).glob('*.png'))  +
                        list(Path(val_mask_dir).glob('*.jpg')))

    if len(val_images) > 100:
        print(f"Using 100 images for evaluation (out of {len(val_images)})")
        val_images = val_images[:100]
        val_masks  = val_masks[:100]

    all_preds = []
    all_targets = []
    ious = []
    precisions = []
    recalls = []

    mean_t = np.array([0.485, 0.456, 0.406])
    std_t  = np.array([0.229, 0.224, 0.225])

    with torch.no_grad():
        for img_path, mask_path in tqdm(zip(val_images, val_masks),
                                        total=len(val_images), desc="Evaluating"):
            img     = Image.open(img_path).convert('RGB').resize(img_size)
            mask    = Image.open(mask_path).convert('L').resize(img_size)
            mask_np = np.array(mask) / 255.0

            img_norm   = (np.array(img).astype(np.float32) / 255.0 - mean_t) / std_t
            img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)

            pred     = model(img_tensor)            # eval → single output
            pred     = torch.sigmoid(pred).cpu().numpy()[0, 0]
            pred_bin = (pred     > 0.5).astype(np.uint8)
            mask_bin = (mask_np  > 0.5).astype(np.uint8)

            h, w          = pred_bin.shape
            total_pixels  = h * w
            if total_pixels > sample_pixels:
                indices   = np.random.choice(total_pixels, sample_pixels, replace=False)
                pred_flat = pred_bin.flatten()[indices]
                mask_flat = mask_bin.flatten()[indices]
            else:
                pred_flat = pred_bin.flatten()
                mask_flat = mask_bin.flatten()

            all_preds.extend(pred_flat)
            all_targets.extend(mask_flat)

            intersection = (pred_bin * mask_bin).sum()
            union        = pred_bin.sum() + mask_bin.sum() - intersection
            ious.append(intersection / (union + 1e-6))

            tp = (pred_bin * mask_bin).sum()
            fp = (pred_bin * (1 - mask_bin)).sum()
            fn = ((1 - pred_bin) * mask_bin).sum()
            precisions.append(tp / (tp + fp + 1e-6))
            recalls.append(tp    / (tp + fn + 1e-6))

    all_preds   = np.array(all_preds,   dtype=np.uint8)
    all_targets = np.array(all_targets, dtype=np.uint8)

    precision = np.mean(precisions)
    recall    = np.mean(recalls)

    return {
        'accuracy'        : accuracy_score(all_targets, all_preds),
        'precision'       : precision,
        'recall'          : recall,
        'f1'              : 2 * (precision * recall) / (precision + recall + 1e-6),
        'mean_iou'        : np.mean(ious),
        'confusion_matrix': confusion_matrix(all_targets, all_preds),
        'n_images'        : len(val_images)
    }


def print_and_plot_metrics(metrics, title, save_path):
    print(f"\n{'='*50}")
    print(f"  {title}")
    print(f"{'='*50}")
    print(f"Pixel Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision      : {metrics['precision']:.4f}")
    print(f"Recall         : {metrics['recall']:.4f}")
    print(f"F1 Score       : {metrics['f1']:.4f}")
    print(f"Mean IoU       : {metrics['mean_iou']:.4f}")
    print(f"{'='*50}")

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(metrics['confusion_matrix'], annot=True, fmt='d', cmap='plasma',
                xticklabels=['Background', 'Foreground'],
                yticklabels=['Background', 'Foreground'],
                cbar_kws={'label': 'Count'},
                ax=ax, linewidths=2, linecolor='#1a1a1a')
    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label',      fontsize=12, fontweight='bold')
    ax.set_title(f'Confusion Matrix — {title}', fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
    plt.show()


# ── 4. Evaluate on training set ───────────────────────────────────────────────
metrics_train = evaluate_validation_set(model, INPUT_DIR, TARGET_DIR, device)
print_and_plot_metrics(metrics_train, "METRICS ON TRAINING DATASET",
                       MODEL_OUTPUT_DIR / 'confusion_matrix_train.png')

# ── 5. Evaluate on validation set ────────────────────────────────────────────
VAL_DIR       = BASE_DIR / "data" / "raw" / "kvasir-sessile" / "sessile-main-Kvasir-SEG"
VAL_INPUT_DIR = VAL_DIR / "images"
VAL_MASK_DIR  = VAL_DIR / "masks"

metrics_val = evaluate_validation_set(model, VAL_INPUT_DIR, VAL_MASK_DIR, device)
print_and_plot_metrics(metrics_val, "METRICS ON VALIDATION DATASET",
                       MODEL_OUTPUT_DIR / 'confusion_matrix_val.png')